# Validación y métricas

Estimación del riesgo de generalización y significado de cada métrica

## Planteamiento

El capítulo 2 mostró que el error de entrenamiento subestima el riesgo del modelo. Seis capítulos después seguimos sin una alternativa rigurosa. Aquí la construimos, y demostramos las tres cosas que casi ningún curso de este nivel demuestra: por qué el error de test es insesgado, por qué deja de serlo en cuanto se usa para elegir, y por qué $R^2$ puede salir negativo.

> **Lo que usamos de antes**
>
> **?@lem-media-optima**, **?@def-fuga**, **?@cor-residuos-ortogonales**, **?@def-lineal-gaussiano**.

## El riesgo de generalización

<span class="theorem-title">**Definición 1 (Riesgo verdadero)**</span> $R(f) = \mathbb{E}\!\left[ (Y - f(X))^2 \right]$, esperanza sobre una observación **nueva** de la misma distribución. Nótese la tipografía: $R$ es el riesgo verdadero, $\hat{R}$ el empírico. Todo el capítulo va de estimar el primero con el segundo.

<span class="theorem-title">**Definición 2 (Entrenamiento, validación y test)**</span> Tres papeles distintos: **entrenar** los parámetros, **elegir** entre modelos, y **reportar** una vez.

<span class="theorem-title">**Teorema 1 (Insesgadez del error de test)**</span> Si (i) $\hat{f}$ es **independiente** del conjunto de test y (ii) los puntos de test son iid de la misma distribución, entonces $\mathbb{E}\!\left[ \hat{R}_{\text{test}}(\hat{f}) \,\vert\,\hat{f} \right] = R(\hat{f})$.

<span class="proof-title">*Demostración*. </span>Linealidad de la esperanza sobre $n_{\mathrm{test}}$ términos idénticamente distribuidos.

> **Important**
>
> La demostración es inmediata; las hipótesis, no. Incumplir (i) es fuga de información (**?@def-fuga**). Incumplir (ii) es lo que ocurre con un anfitrión que tiene doce anuncios (<a href="#sec-grupos" class="quarto-xref">Sección 5</a>).

<span class="theorem-title">**Proposición 1 (Precisión del error de test)**</span> $\mathrm{Var}\!\left( \hat{R}_{\text{test}} \right) = v/n_{\mathrm{test}}$ con $v$ la varianza de la pérdida puntual; el error estándar es $\sqrt{v/n_{\mathrm{test}}}$. De aquí se responde cuántos datos de test hacen falta.

## Por qué el test se toca una vez

<span class="theorem-title">**Teorema 2 (Sesgo de selección)**</span> $\displaystyle \mathbb{E}\!\left[ \min_m \hat{R}_m \right] \leq \min_m \mathbb{E}\!\left[ \hat{R}_m \right]$. Elegir el modelo con menor error de test hace que **ese número** sea optimista.

<span class="proof-title">*Demostración*. </span>

<span class="proof-title">*Observación 1* (Consecuencias). </span>Los hiperparámetros se eligen en **validación**; el test se toca **una vez**. Esto explica por qué las tablas de clasificación públicas se degradan con el tiempo, y por qué el proyecto exige por escrito que el test se use una sola vez.

## Validación cruzada

<span class="theorem-title">**Definición 3 (Validación cruzada en $K$ bloques)**</span> $\displaystyle \mathrm{VC}_K = \frac{1}{K}\sum_{k=1}^{K}\hat{R}_k$, con $\hat{R}_k$ el riesgo empírico sobre el bloque $k$ del modelo entrenado sin él.

> **Una precisión sobre el promedio**
>
> $\frac{1}{K}\sum_k(\text{media del bloque})$ coincide con $\frac{1}{n}\sum_i \ell_i$ **solo si los bloques tienen el mismo tamaño**. `cross_val_score` hace lo primero.

<span class="theorem-title">**Proposición 2 (Sesgo de la validación cruzada)**</span> Cada modelo se entrena con $n(K-1)/K$ observaciones, luego $\mathrm{VC}_K$ estima el riesgo de un modelo entrenado con **menos** datos que el que finalmente se despliega: el sesgo es **pesimista** y decrece con $K$.

<span class="theorem-title">**Lema 1 (Varianza de una media de variables correladas)**</span> Si $Z_1,\dots,Z_K$ tienen varianza $v$ y correlación $\rho$ dos a dos, entonces $$\mathrm{Var}\!\left( \frac{1}{K}\sum_k Z_k \right) = \rho v + \frac{(1-\rho)v}{K}.$$

<span class="proof-title">*Demostración*. </span>

<span class="theorem-title">**Corolario 1 (Por qué $K=5$ o $K=10$)**</span> Al crecer $K$ el sesgo baja (<a href="#prp-sesgo-cv" class="quarto-xref">Proposición 2</a>), pero los conjuntos de entrenamiento se solapan más, $\rho$ crece y por 1 la varianza **no baja**: tiende a $\rho v$. Con $K=n$ (LOOCV) el sesgo es mínimo, la varianza no se promedia y el coste es $n$ ajustes. **No hay ningún teorema que fije $K$**: es un compromiso empírico, y decirlo es contenido, no laguna.

> **Important**
>
> El lema explica por qué promediar tiene un suelo: por mucho que crezca $K$, la varianza no baja de $\rho v$. Es el mismo argumento por el que, en los métodos que promedian modelos, hace falta que esos modelos estén poco correlacionados entre sí.

## Cuando la hipótesis iid es falsa

<span class="proof-title">*Observación 2* (Grupos). </span>Un anfitrión con doce anuncios rompe la hipótesis (ii) de <a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a>: hay filas de test dependientes de filas de entrenamiento. `GroupKFold` sobre `host_id` restaura la hipótesis, y la diferencia entre las estimaciones de error de ambos esquemas cuantifica el efecto de la dependencia entre filas.

## Métricas

<span class="theorem-title">**Definición 4 ($\mathrm{EAM}$, $\mathrm{ECM}$, $\mathrm{RECM}$)**</span> Con unidades: si $y$ está en euros, $\mathrm{ECM}$ está en euros al cuadrado; $\mathrm{RECM}$ y $\mathrm{EAM}$, en euros. Un $\mathrm{ECM}$ de 3.400 no es interpretable si no se indican las unidades de $y$.

<span class="theorem-title">**Proposición 3 ($\mathrm{RECM}\geq \mathrm{EAM}$ siempre)**</span>  

<span class="proof-title">*Demostración*. </span>

<span class="proof-title">*Observación 3* (Elegir métrica es elegir modelo). </span>$\mathrm{ECM}$ apunta a la **media** (**?@lem-media-optima**); $\mathrm{EAM}$, a la **mediana** (**?@prp-laplace-mae**). No son dos formas de medir lo mismo: estiman funcionales distintos de la distribución condicional.

## $R^2$

<span class="theorem-title">**Definición 5 ($\mathrm{SCE}$, $\mathrm{SCT}$ y $R^2$)**</span> $\mathrm{SCE}= \sum_i(y_i-\hat{y}_i)^2$, $\mathrm{SCT}= \sum_i(y_i-\bar{y})^2$, $\ R^2= 1 - \mathrm{SCE}/\mathrm{SCT}$.

<span class="theorem-title">**Teorema 3 ($R^2$ compara contra el modelo de la media)**</span> $\mathrm{SCT}= n\,\hat{R}(\bar{f})$ donde $\bar{f}\equiv \bar{y}$ es **el mejor modelo constante** (**?@lem-media-optima**). Por tanto $$R^2= 1 - \frac{\hat{R}(\hat{f})}{\hat{R}(\bar{f})}:$$ la fracción del riesgo del modelo nulo que el modelo consigue eliminar. $R^2=0$ significa “igual de bueno que la media”; $R^2=1$, predicción perfecta.

<span class="theorem-title">**Teorema 4 ($\mathrm{SCT}= \mathrm{SCReg}+ \mathrm{SCE}$ **en la muestra**)**</span> Con mínimos cuadrados, término independiente y **sobre los datos de entrenamiento**, $\sum_i(y_i-\bar{y})^2 = \sum_i(\hat{y}_i-\bar{y})^2 + \sum_i(y_i-\hat{y}_i)^2$, de donde $0 \le R^2\le 1$.

<span class="proof-title">*Demostración*. </span>

<span class="theorem-title">**Teorema 5 (Fuera de la muestra $R^2$ puede ser negativo)**</span> En test **falla **?@cor-residuos-ortogonales****, luego falla <a href="#thm-sct-descompone" class="quarto-xref">Teorema 4</a> y $R^2$ no está acotado inferiormente. Además $$R^2_{\text{test}} < 0 \iff \mathrm{SCE}_{\text{test}} > \mathrm{SCT}_{\text{test}} \iff \text{el modelo predice peor que la constante } \bar{y}.$$

<span class="proof-title">*Demostración*. </span>

$R^2$`r2_score` usa la media del vector $y$ que se le pasa, es decir la de **test**. $R^2$ es adimensional, luego **no es comparable entre conjuntos de datos** porque $\mathrm{SCT}$ depende de $\mathrm{Var}\!\left( y \right)$. Y un $R^2$ alto no implica un modelo útil.

## Laboratorio

## Error frecuente

**“He probado veinte modelos y me quedo con el mejor en test”**

Ese “mejor en test” ya no estima nada, por <a href="#thm-sesgo-seleccion" class="quarto-xref">Teorema 2</a>. Con veinte modelos y un test pequeño, el ganador lo es en buena parte por suerte. Hiperparámetros en validación; test, una vez.

## Notación ↔ código

| Matemáticas | Código |
|------------------------------------|------------------------------------|
| $\mathrm{VC}_K$ | `cross_val_score(pipe, X, y, cv=KFold(5, shuffle=True, random_state=42))` |
| $\mathrm{RECM}$ | `scoring="neg_root_mean_squared_error"` (sklearn devuelve el valor con signo negativo) |
| $R^2$ | `r2_score(y_test, yhat)` |
| grupos | `GroupKFold(n_splits=5).split(X, y, groups=host_id)` |

## Ejercicios

<span class="theorem-title">**Ejercicio 1**</span> Sobre una tabla de seis filas, calcule a mano $\mathrm{EAM}$, $\mathrm{ECM}$, $\mathrm{RECM}$ y $R^2$. Verifique <a href="#prp-rmse-mae" class="quarto-xref">Proposición 3</a>.

<span class="theorem-title">**Ejercicio 2**</span> Construya tres puntos y un modelo lineal con $R^2_{\text{test}} < 0$. Explique en una frase qué le está diciendo ese número a quien lo lea.

<span class="theorem-title">**Ejercicio 3**</span> Dadas las pérdidas de tres bloques $(12{,}1;\ 9{,}8;\ 14{,}0)$ de tamaños $(40, 40, 41)$, calcule $\mathrm{VC}_3$ de las dos maneras de <a href="#def-cv" class="quarto-xref">Definición 3</a> y explique la diferencia.

## Resumen

- El error de test es insesgado **solo** bajo dos hipótesis explícitas (<a href="#thm-test-insesgado" class="quarto-xref">Teorema 1</a>), y elegir con él las rompe (<a href="#thm-sesgo-seleccion" class="quarto-xref">Teorema 2</a>).
- $K=5$ o $10$ no es un teorema: es sesgo (<a href="#prp-sesgo-cv" class="quarto-xref">Proposición 2</a>) contra varianza
  1.  contra coste.
- $R^2$ es una comparación contra la media (<a href="#thm-r2-comparacion" class="quarto-xref">Teorema 3</a>); fuera de la muestra puede ser negativo, y eso significa algo muy concreto (<a href="#thm-r2-negativo" class="quarto-xref">Teorema 5</a>).